# Example 16.10 / 1: Compare Two Starts for Five Points

Companion notebook for **16.10 Pairwise Distance Ratio for N Points** in the AMPL lab Documentation, section "Compare Two Starts for Five Points". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Compare Two Starts for Five Points

The model below is the scale-normalized formulation above. For each seed, the run sets `ampl.option["randseed"]`, draws a random start with `ampl.eval` because AMPL's `Uniform` function has no amplpy counterpart, resets the start of `min_sq_distance` to 0 with `set_value`, solves with the local solver Ipopt, reads `x` with `to_dict()`, and recomputes every squared distance in Python:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param N integer >= 2;  # number of points
param D integer >= 1;  # space dimension
set POINTS := 1..N;  # points to place
set DIMS := 1..D;  # coordinate axes

var x {POINTS, DIMS} >= 0, <= 1;  # coordinates in the unit cube
var min_sq_distance >= 0, <= 1;  # smallest squared pairwise distance t

subject to DistanceLower {i in POINTS, j in POINTS: i < j}:  # every pair at least t apart
    sum {k in DIMS} (x[i,k]-x[j,k])^2 >= min_sq_distance;
subject to DistanceUpper {i in POINTS, j in POINTS: i < j}:  # largest squared distance at most 1
    sum {k in DIMS} (x[i,k]-x[j,k])^2 <= 1;

maximize Spread: min_sq_distance;  # max/min distance ratio is 1/sqrt(t)

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["N"] = 5
ampl.param["D"] = 2

In [5]:
# run.py
import math

for seed in [1, 2]:
    ampl.option["randseed"] = seed
    ampl.eval("let {i in POINTS, k in DIMS} x[i,k] := Uniform(0, 1);")
    ampl.var["min_sq_distance"].set_value(0)
    ampl.solve(solver="ipopt", verbose=False)
    x = ampl.var["x"].to_dict()
    d = [
        sum((x[i, k] - x[j, k]) ** 2 for k in (1, 2))
        for i in range(1, 6)
        for j in range(i + 1, 6)
    ]
    t = ampl.obj["Spread"].value()
    print(
        f"seed {seed}: {ampl.solve_result}, t = {t:.6f}, min = {min(d):.6f}, "
        f"max = {max(d):.6f}, ratio = {math.sqrt(max(d) / min(d)):.6f}"
    )

seed 1: solved, t = 0.283119, min = 0.283119, max = 1.000000, ratio = 1.879385
seed 2: solved, t = 0.381966, min = 0.381966, max = 1.000000, ratio = 1.618034


### Another way in amplpy

Both runs report `solved`, but they are different local optima. Seed 2 reaches $t=0.381966$, a distance ratio of $1/\sqrt{t}\approx1.618$, while seed 1 stops at $t=0.283119$. The recomputed minimum equals `min_sq_distance` and the maximum is 1, as the normalization argument predicts. Resetting `min_sq_distance` matters: in a test run without that line, the second solve started from the first solution's $t=0.283119$ and returned 0.283119 for seed 2 as well. The run below, with the same model.mod and data.py, repeats the seeds 1 to 8 for five points and then solves the small cases of the table with a helper `solve_from`, which draws a random start or sets every coordinate to the same value:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
ampl.param["N"] = 5
ampl.param["D"] = 2

In [8]:
# run.py
def solve_from(n, d, seed=None, start=None):
    ampl.param["N"] = n
    ampl.param["D"] = d
    if start is None:
        ampl.option["randseed"] = seed
        ampl.eval("let {i in POINTS, k in DIMS} x[i,k] := Uniform(0, 1);")
    else:
        ampl.eval(f"let {{i in POINTS, k in DIMS}} x[i,k] := {start};")
    ampl.var["min_sq_distance"].set_value(0)
    ampl.solve(solver="ipopt", verbose=False)
    return ampl.solve_result, round(ampl.obj["Spread"].value(), 6)


print("N = 5, seeds 1-8:", [solve_from(5, 2, seed=s)[1] for s in range(1, 9)])
print("N = 3, D = 2:", solve_from(3, 2, seed=1))
print("N = 4, D = 2:", solve_from(4, 2, seed=1))
print("N = 3, D = 1:", solve_from(3, 1, seed=1))
print("N = 3, D = 1, x:", [round(v, 6) for v in ampl.var["x"].to_dict().values()])
print("N = 4, D = 2, all x start at 0:", solve_from(4, 2, start=0))

N = 5, seeds 1-8: [0.283119, 0.381966, 0.283119, 0.283119, 0.381966, 0.381966, 0.283119, 0.381966]
N = 3, D = 2: ('solved', 1.0)
N = 4, D = 2: ('solved', 0.5)
N = 3, D = 1: ('solved', 0.25)
N = 3, D = 1, x: [0.5, 0.0, 1.0]
N = 4, D = 2, all x start at 0: ('solved', 0.0)


### Pure AMPL: data.dat

Four of the eight seeds return 0.381966 and four stop at 0.283119, each with status `solved`. The row marked "Gurobi licence" is quoted from a run with Gurobi's global option; the others come from the runs on this page:

| Item | Concrete instance | Result |
|---|---|---|
| equilateral optimum | $N=3$, $D=2$, random start | $t=1$, ratio 1 |
| square configuration | $N=4$, $D=2$, random start | $t=0.5$, ratio $\sqrt2\approx1.414$ |
| one dimension | $N=3$, $D=1$, randseed 1 | $t=0.25$: points at 0, 0.5 and 1, ratio 2 |
| coincident start, `min_sq_distance = 0` | $N=4$, $D=2$, every `x[i,k]` started at 0 | `solved` with $t=0$: a stationary point with no finite ratio, rejected by validation |
| time-limited global solve (Gurobi licence) | $N=4$, $D=2$, Gurobi with `gurobi_options 'global=1 timelim=3'` | `solve_result = limit`, best arrangement found $t=0.500003$; report it with its bound, not as `solved` |

For pure AMPL, the five-point data in AMPL data format; with this data.dat, the first run above prints the same output. In pure AMPL, `option randseed 2;`, `let {i in POINTS, k in DIMS} x[i,k] := Uniform(0, 1);`, `option solver ipopt;` and `solve;` do the same work for one seed, and Ipopt then also prints its banner and log:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
%%writefile data.dat
param N := 5;
param D := 2;

Writing data.dat


In [11]:
%%ampl_eval
data data.dat;

In [12]:
# run.py
import math

for seed in [1, 2]:
    ampl.option["randseed"] = seed
    ampl.eval("let {i in POINTS, k in DIMS} x[i,k] := Uniform(0, 1);")
    ampl.var["min_sq_distance"].set_value(0)
    ampl.solve(solver="ipopt", verbose=False)
    x = ampl.var["x"].to_dict()
    d = [
        sum((x[i, k] - x[j, k]) ** 2 for k in (1, 2))
        for i in range(1, 6)
        for j in range(i + 1, 6)
    ]
    t = ampl.obj["Spread"].value()
    print(
        f"seed {seed}: {ampl.solve_result}, t = {t:.6f}, min = {min(d):.6f}, "
        f"max = {max(d):.6f}, ratio = {math.sqrt(max(d) / min(d)):.6f}"
    )

seed 1: solved, t = 0.283119, min = 0.283119, max = 1.000000, ratio = 1.879385
seed 2: solved, t = 0.381966, min = 0.381966, max = 1.000000, ratio = 1.618034
